# Data modeling
Uses the shared logic in `src/` so the notebook and the app follow the same rules (see `assumptions.md`).

In [1]:
import pandas as pd

from src.loader import load_shifts, load_holidays, load_payroll
from src.hours import parse_shifts, attribute_hours, data_cutoff, quality_report
from src.integrity import find_duplicate_people, add_person_key, find_overlaps

shifts_raw = load_shifts("data/shifts.csv")
holidays = load_holidays("data/public_holidays.csv")
employees = pd.read_csv("data/employees.csv")
sites = pd.read_csv("data/sites.csv")
notes = pd.read_csv("data/shift_notes.csv")
weekly_summary = pd.read_csv("data/weekly_summary.csv")

shifts_raw.head()

,shift_id,employee_id,site_id,shift_date,clock_in_time,clock_out_time
0,S100048,E1002,ST-02,2026-06-08,07:00,15:15
1,S100191,E1005,ST-05,2026-06-08,08:00,16:15
2,S100355,E1009,ST-02,2026-06-08,08:00,20:15
3,S100449,E1011,ST-01,2026-06-08,09:30,19:30
4,S100574,E1014,ST-03,2026-06-08,06:00,17:30


## Parse shifts and data quality
Overnight = clock-out earlier than clock-in (+24h). Empty/invalid times are flagged and excluded. Shifts over 12 hours are flagged but counted.

In [2]:
shifts = parse_shifts(shifts_raw)
quality_report(shifts)

{'shifts': 8863,
 'overnight': 1010,
 'over_12h_flagged': 1279,
 'invalid_date': 0,
 'invalid_clock_in': 0,
 'invalid_clock_out': 184,
 'zero_duration': 0,
 'excluded': 184}

In [3]:
assert (shifts.loc[~shifts.excluded, "duration_hours"] > 0).all()
print("shortest / longest counted shift:", shifts.duration_hours.min(), shifts.duration_hours.max())
print("flagged invalid (excluded):", shifts.excluded.sum())
shifts[shifts.excluded][["shift_id", "employee_id", "shift_date", "clock_in_time", "clock_out_time"]].head()

shortest / longest counted shift: 3.5 13.5
flagged invalid (excluded): 184


,shift_id,employee_id,shift_date,clock_in_time,clock_out_time
138,S100245,E1006,2026-06-09,07:15,NaN
154,S101273,E1030,2026-06-09,07:00,NaN
176,S102834,E1065,2026-06-09,19:30,NaN
237,S107084,E1162,2026-06-09,09:00,NaN
250,S107993,E1182,2026-06-09,18:00,NaN


## Attribute hours to a day and week (greater-portion rule)
A shift crossing midnight goes whole to the day with the greater portion; an exact tie is split at midnight. The attributed day sets the week and the 2x Sunday / public-holiday rate.

In [4]:
segments = attribute_hours(shifts, holidays)

cutoff = data_cutoff(shifts)
current_week = cutoff - pd.Timedelta(days=cutoff.weekday())
print("data ends", cutoff.date(), cutoff.day_name(), "-> in-progress week starts", current_week.date())
print("segments:", len(segments), "| split ties:", len(segments) - (~shifts.excluded).sum())
print("2x segments (Sunday or public holiday):", (segments.multiplier == 2.0).sum())

data ends 2026-08-12 Wednesday -> in-progress week starts 2026-08-10
segments: 8726 | split ties: 47
2x segments (Sunday or public holiday): 1463


## Integrity flags (potential fraud, for escalation)
**Duplicate people:** employee IDs sharing an ID number, bank account or tax number. Bank and tax values are compared in memory only and are never shown. **Overlapping shifts:** one person at two sites at the same time. Overlapping hours are still counted; they are only flagged.

In [5]:
payroll = load_payroll("data/payroll_details.csv")   # employee_id, account_number, tax_number only
people = find_duplicate_people(employees, payroll)
del payroll                                          # not kept around

print("duplicate people:", len(people), "| escalate:", (people.severity == "Escalate").sum())
people[["person_key", "employee_ids", "names", "evidence", "severity"]]

duplicate people: 5 | escalate: 5


,person_key,employee_ids,names,evidence,severity
0,E1035,"[E1035, E1036]","[Tshepo Motaung, T. Motaung]",same ID number; same bank account; same tax nu...,Escalate
1,E1090,"[E1090, E1091]","[Pieter Van Wyk, P. Van Wyk]",same ID number; same bank account; same tax nu...,Escalate
2,E1097,"[E1097, E1098]","[Busisiwe Jacobs, B. Jacobs]",same ID number; same bank account; same tax nu...,Escalate
3,E1126,"[E1126, E1127]","[Lerato Motaung, L. Motaung]",same ID number; same bank account; same tax nu...,Escalate
4,E1193,"[E1193, E1194]","[Lucky Sibiya, L. Sibiya]",same ID number; same bank account; same tax nu...,Escalate


In [6]:
overlaps = find_overlaps(shifts, sites, people)

print("overlapping shift pairs:", len(overlaps))
print(overlaps.groupby("kind").agg(pairs=("shift_id_a", "count"),
                                   people=("person_key", "nunique"),
                                   different_province=("different_province", "sum"),
                                   overlap_hours=("overlap_hours", "sum")))
print("high severity (different provinces):", (overlaps.severity == "High").sum())
overlaps[["person_key", "kind", "site_a", "site_b", "start_a", "end_a", "start_b", "end_b", "overlap_hours", "severity"]].head(8)

overlapping shift pairs: 195
          pairs  people  different_province  overlap_hours
kind                                                      
cross_id     69       5                  31         510.00
same_id     126      81                  99         432.75
high severity (different provinces): 130


,person_key,kind,site_a,site_b,start_a,end_a,start_b,end_b,overlap_hours,severity
0,E1005,same_id,ST-05,ST-04,2026-07-01 06:00:00,2026-07-01 14:15:00,2026-07-01 08:00:00,2026-07-01 12:00:00,4.00,High
1,E1006,same_id,ST-02,ST-04,2026-06-16 06:00:00,2026-06-16 10:00:00,2026-06-16 06:00:00,2026-06-16 18:00:00,4.00,High
2,E1006,same_id,ST-04,ST-01,2026-06-22 06:00:00,2026-06-22 17:15:00,2026-06-22 08:30:00,2026-06-22 12:30:00,4.00,High
3,E1007,same_id,ST-06,ST-02,2026-07-04 06:30:00,2026-07-04 19:00:00,2026-07-04 09:00:00,2026-07-04 13:00:00,4.00,High
4,E1015,same_id,ST-05,ST-06,2026-07-14 08:00:00,2026-07-14 17:45:00,2026-07-14 09:15:00,2026-07-14 13:15:00,4.00,High
5,E1015,same_id,ST-06,ST-05,2026-07-17 06:00:00,2026-07-17 10:00:00,2026-07-17 06:15:00,2026-07-17 15:15:00,3.75,High
6,E1016,same_id,ST-05,ST-04,2026-06-26 06:00:00,2026-06-26 10:00:00,2026-06-26 06:00:00,2026-06-26 16:15:00,4.00,High
7,E1019,same_id,ST-05,ST-02,2026-08-05 06:15:00,2026-08-05 15:45:00,2026-08-05 08:15:00,2026-08-05 12:15:00,4.00,High


## Person-level weekly hours
Hours are summed across all employee IDs of one person (recorded hours, overlaps counted), because splitting hours across two IDs is what hides a breach.

In [7]:
segments = add_person_key(segments, people)
complete_weeks = segments["week_start"] < current_week

per_id = segments.groupby(["employee_id", "week_start"])["hours"].sum()
per_person = segments.groupby(["person_key", "week_start"])["hours"].sum()

breach_id = ((per_id - 45) > 10)[per_id.index.get_level_values(1) < current_week]
breach_person = ((per_person - 45) > 10)[per_person.index.get_level_values(1) < current_week]
print("breach-weeks, per employee ID :", breach_id.sum())
print("breach-weeks, per person      :", breach_person.sum())

dup_keys = list(people.person_key)
per_person.loc[dup_keys].unstack(0).round(1)

breach-weeks, per employee ID : 87
breach-weeks, per person      : 114


person_key,E1035,E1090,E1097,E1126,E1193
week_start,,,,,
2026-06-08,62.8,66.8,65.5,51.5,53.0
2026-06-15,54.5,56.5,53.5,56.0,46.8
2026-06-22,61.2,54.2,66.2,60.2,55.2
2026-06-29,41.2,58.8,55.5,61.8,54.8
2026-07-06,40.2,57.8,66.0,56.8,50.0
2026-07-13,52.5,53.5,51.8,71.8,57.0
2026-07-20,59.5,63.5,67.0,46.5,57.0
2026-07-27,51.8,53.2,55.2,56.0,59.8
2026-08-03,60.2,53.2,52.5,65.2,66.5


## Compare with the client's `weekly_summary.csv`
The client's totals follow a different rule (clock-in date, missing clock-outs as 0, no person merge), so some employee-weeks should differ.

In [8]:
weekly = segments.groupby(["employee_id", "week_start"])["hours"].sum().reset_index()

cmp = weekly_summary.assign(week_starting=pd.to_datetime(weekly_summary.week_starting)).merge(
    weekly, left_on=["employee_id", "week_starting"], right_on=["employee_id", "week_start"], how="outer")
cmp["diff"] = cmp["hours"].fillna(0) - cmp["total_hours"].fillna(0)
print("employee-weeks:", len(cmp), "| differ from client summary:", (cmp["diff"].abs() > 0.01).sum())
cmp[cmp["diff"].abs() > 0.01].sort_values("diff", key=abs, ascending=False).head(10)[["employee_id", "week_starting", "total_hours", "hours", "diff"]]

employee-weeks: 2124 | differ from client summary: 122


,employee_id,week_starting,total_hours,hours,diff
34,E1004,2026-07-06,46.25,32.75,-13.5
850,E1086,2026-06-15,49.25,35.75,-13.5
1797,E1181,2026-06-22,37.50,51.00,13.5
1796,E1181,2026-06-15,47.75,34.25,-13.5
778,E1078,2026-08-10,13.00,26.50,13.5
1581,E1159,2026-07-13,43.50,57.00,13.5
1580,E1159,2026-07-06,37.25,23.75,-13.5
802,E1081,2026-06-29,35.50,22.00,-13.5
1456,E1146,2026-08-10,25.25,38.75,13.5
819,E1083,2026-06-08,38.00,24.50,-13.5


## People averaging more than 45 hours a week
Per person (all their IDs combined), complete Mon-Sun weeks only.

In [9]:
person_weekly = per_person.reset_index()
complete = person_weekly[person_weekly["week_start"] < current_week]
avg_hours = complete.groupby("person_key")["hours"].mean().rename("avg_weekly_hours")
over_45 = avg_hours[avg_hours > 45].sort_values(ascending=False).reset_index()
over_45["merged_duplicate"] = over_45["person_key"].isin(dup_keys)
print("people averaging over 45 hours/week:", len(over_45), "| of which merged duplicates:", over_45.merged_duplicate.sum())
over_45.merge(employees[["employee_id", "full_name", "role", "primary_site_id"]], left_on="person_key", right_on="employee_id").drop(columns="employee_id").head(20)

people averaging over 45 hours/week: 59 | of which merged duplicates: 5


,person_key,avg_weekly_hours,merged_duplicate,full_name,role,primary_site_id
0,E1097,59.250000,True,Busisiwe Jacobs,Supervisor,ST-02
1,E1126,58.416667,True,Lerato Motaung,Supervisor,ST-06
2,E1090,57.500000,True,Pieter Van Wyk,Cleaner,ST-05
3,E1193,55.555556,True,Lucky Sibiya,Kitchen Assistant,ST-03
4,E1035,53.777778,True,Tshepo Motaung,Security Guard,ST-01
5,E1039,52.694444,False,Pieter Naidoo,Kitchen Assistant,ST-04
6,E1122,52.111111,False,Dineo Tshabalala,Kitchen Assistant,ST-03
7,E1061,51.583333,False,Pieter Baloyi,Supervisor,ST-01
8,E1198,51.166667,False,Andile Jacobs,Supervisor,ST-04
9,E1184,50.333333,False,Portia Motaung,Security Guard,ST-06


## Small prediction for the in-progress week
Projected final hours = hours so far this week + the person's average Thu-Sun hours (by clock-in day) from earlier weeks, all of a person's IDs combined. Breach if projected overtime is more than 10 hours (over 55 total). Both IDs of a merged person get the same prediction. A first pass: it ignores that 2x hours may count differently.

In [10]:
seg = segments.merge(shifts[["shift_id", "shift_date"]], on="shift_id")
seg["start_weekday"] = seg["shift_date"].dt.weekday  # day the shift was clocked in, Mon=0

so_far = seg[seg["week_start"] == current_week].groupby("person_key")["hours"].sum().rename("hours_so_far")

rest = (seg[(seg["week_start"] < current_week) & (seg["start_weekday"] >= 3)]
        .groupby(["person_key", "week_start"])["hours"].sum().reset_index()
        .groupby("person_key")["hours"].mean().rename("typical_rest_of_week"))

person_ids = add_person_key(employees[["employee_id", "full_name"]], people)
persons = pd.DataFrame({"person_key": person_ids.person_key.unique()}).set_index("person_key").join([so_far, rest]).fillna(0)
persons["projected_hours"] = persons["hours_so_far"] + persons["typical_rest_of_week"]
persons["projected_overtime"] = (persons["projected_hours"] - 45).clip(lower=0)
persons["will_breach"] = (persons["projected_overtime"] > 10).astype(int)

pred = person_ids.merge(persons, left_on="person_key", right_index=True)
print("predicted breaches: people", persons["will_breach"].sum(), "| employee IDs", pred["will_breach"].sum(), "of", len(pred))
pred.sort_values("projected_hours", ascending=False).head(15)

predicted breaches: people 33 | employee IDs 38 of 213


,employee_id,full_name,person_key,hours_so_far,typical_rest_of_week,projected_hours,projected_overtime,will_breach
98,E1099,Portia Fourie,E1099,51.75,26.055556,77.805556,32.805556,1
125,E1126,Lerato Motaung,E1126,38.75,36.027778,74.777778,29.777778,1
126,E1127,L. Motaung,E1126,38.75,36.027778,74.777778,29.777778,1
131,E1132,Kabelo Fourie,E1132,38.50,28.138889,66.638889,21.638889,1
193,E1194,L. Sibiya,E1193,28.25,37.972222,66.222222,21.222222,1
192,E1193,Lucky Sibiya,E1193,28.25,37.972222,66.222222,21.222222,1
41,E1042,Pieter Khumalo,E1042,36.25,28.583333,64.833333,19.833333,1
103,E1104,Given Tshabalala,E1104,34.75,29.694444,64.444444,19.444444,1
60,E1061,Pieter Baloyi,E1061,28.25,35.416667,63.666667,18.666667,1
86,E1087,Mandla Mthembu,E1087,35.50,27.138889,62.638889,17.638889,1
